https://github.com/mCodingLLC/VideosSampleCode/tree/master/videos/135_modern_logging

Logger 
- Filters  
- Formatters  
- Handlers  
- Loggers  
  
<img src="images/logger.png" width="400">
  
LOG RECORD  
- message  
- level  
- created  
- thread  
  
Log record is a python object. Formatter converts it to string.  
Log record gets passed to each handler.  

# Tip  
Put all handlers/filters on the root logger.  
Don't use the root logger in your code.  
logger = logging.getLogger("my_app") # Use your own logger  
  
small application - use one logger.  
large application - One logger per major-subcomponent  
dont -> getLogger(__name__) in every file. # The logger is global that live for the entire life of the program.  
  
Store config in JSON or YAML file.  
Add context with logging.info  
Use a Queue handler to log off the main thread.  
For libraries dont configure logging.  

In [ ]:
import logging.config
import json
import pathlib as pathLib

def setup_logging():
    config_file = pathLib.Path("logging_config.json")
    with open(config_file, "r") as f:
        config = json.load(f)
    logging.config.dictConfig(config)
    queue_handler = logging.getHandlerByName("queue_handler")
    if queue_handler is not None:
        queue_handler.listener.start()
        atexit.register(queue_handler.listener.stop)

In [ ]:
import logging.config

# if you want to customize your own format -> docs.python.org/3/library/logging.html
logging_config = {
    "version": 1, 
    "disable_existing_loggers": False,
    "formatters": {
        "simple": {
            "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
            "datefmt": "%Y-%m-%d T%H:%M:%Sz",
        },
        
    },
    "handlers": {
        "stdout": {
            "class": "logging.StreamHandler",
            "formatter": "simple",
            "stream": "ext://sys.stdout" # sys.stdout a variable defined outside of this config.
        },
        "file": {
            "class": "logging.handlers.RotatingFileHandler",
            "level": "DEBUG",
            "formatter": "simple",
            "filename": "discord_bot.log",
            "maxBytes": 1024*1024, # 1 MB
            "backupCount": 3, # once file reaches 1 MB, it will be renamed to discord_bot.log.1, and a new discord_bot.log will be created. This will keep up to 3 backup files.
        },
        "queue_handler": {
            "class": "logging.handlers.QueueHandler",
            "handlers": ["stdout", "file"],
            "respect_handler_level": True, # This ensures that the handler will only emit records that are at or above the handler's level. If False, it will emit all records regardless of level.
        }
    },
    "loggers": {
        "root": {
            "handlers": ["stdout"],
            "level": "DEBUG",
            "propagate": False
        }
    }
}



